# Is a language model secretly doing latent-variable inference?
### CS771 Mini-Project 2, Problem 2 — latent variable models, neural networks and LLMs in one toy world

Large language models are trained only to predict the next token. Yet they appear to "work out" what kind of text
they are looking at — a recipe, a proof, Python code — and adapt their predictions to it, without any change to their
weights. This is called **in-context learning**. In this lab you will build a tiny world in which you *know* the right
answer, and find out what a small transformer trained on next-token prediction is actually computing.

**The world.** Text over a vocabulary of $V=8$ tokens. Before writing a sequence, nature secretly picks a **concept**
$z\in\{1,\dots,4\}$; each concept is a Markov chain (see below) with its own transition matrix $\mathbf{A}_z$. The data are
therefore a **mixture of Markov chains** — a latent variable model, just like a GMM, but with discrete sequences as
observations.

**Plan** (about 3 hours of focused work; all heavy code is provided)

| part | what you do | time |
|---|---|---|
| 1 | Latent-variable inference: the posterior over the concept and the Bayes-optimal predictor | 30 min |
| 2 | Learn the mixture from unlabelled text with EM | 45 min |
| 3 | Train a tiny transformer; compare its loss with the best possible loss | 25 min |
| 4 | In-context learning as posterior inference | 20 min |
| 5 | Look inside: can the posterior be read off the hidden states? | 20 min |
| 6 | Limits: new concepts, and what changes with more diverse training data | 25 min |

---
### Background you need (a few ideas not covered in the lectures)

* **Markov chain.** A model of a token sequence in which the next token depends only on the *current* token, through a
  lookup table $\mathbf{A}$: $A[a,b] = p(x_{t+1}=b\mid x_t=a)$. Each row of $\mathbf{A}$ is a probability distribution over the $V$
  tokens (a multinoulli). The probability of a whole sequence is a product of table entries:
  $p(x_1,\dots,x_T) = p(x_1)\prod_{t=2}^{T}A[x_{t-1},x_t]$. It is the simplest possible "language model": a bigram model.
* **Entropy** of a distribution $p$ over $K$ outcomes: $H(p) = -\sum_k p_k\log p_k$. It is $0$ when $p$ is certain (one-hot) and
  $\log K$ when $p$ is uniform. Cross-entropy (Lectures 7, 19) is its two-distribution cousin.
* **Perplexity** of a language model: $\exp(\text{average cross-entropy per token, in nats})$. It reads as "the number of
  equally likely tokens the model is effectively choosing between": a uniform guess over 8 tokens has perplexity 8.
* **$R^2$** (fraction of variance explained) of a regression: $R^2 = 1 - \frac{\sum_i\|\mathbf{y}_i-\hat{\mathbf{y}}_i\|^2}{\sum_i\|\mathbf{y}_i-\bar{\mathbf{y}}\|^2}$.
  $1$ is a perfect fit, $0$ is no better than predicting the mean.
* **Dirichlet prior** (used only in Part 6): the Beta–Bernoulli result of Lecture 12, extended from 2 outcomes to $V$.
  With a Beta$(\alpha,\beta)$ prior, the posterior just adds the observed counts to the pseudo-counts $\alpha,\beta$; with a
  Dirichlet$(\alpha,\dots,\alpha)$ prior on a multinoulli over $V$ outcomes, the posterior adds the counts of each outcome to
  $\alpha$, and the posterior predictive is $(\text{count}_b+\alpha)/(\text{total}+V\alpha)$.

Cells marked `# YOUR CODE HERE` are yours. **Predict first** boxes ask for a prediction *before* you run a cell; write
it down honestly — you are graded on the reasoning, not on being right. Runs on Colab in a few minutes (a GPU makes the
two training runs take seconds; on CPU each takes 2–4 minutes; EM takes under a minute).

In [ ]:
import time, math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import Ridge

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

V, K, T = 8, 4, 64                                   # vocabulary size, number of concepts, sequence length
rng = np.random.default_rng(0)
A = rng.dirichlet(0.3 * np.ones(V), size=(K, V))     # A[k, a, b] = p(next token b | current token a, concept k)
LOG_PRIOR = np.log(np.full(K, 1.0 / K))              # concepts are equally likely

def sample_sequences(n, A, rng):
    """Draws n sequences: concept z ~ uniform, first token ~ uniform, then the Markov chain A[z]. Returns X (n, T), z (n,)."""
    z = rng.integers(0, len(A), n)
    X = np.zeros((n, T), dtype=np.int64)
    X[:, 0] = rng.integers(0, V, n)
    for t in range(1, T):
        P = A[z, X[:, t - 1]]
        X[:, t] = (P.cumsum(1) < rng.random((n, 1))).sum(1).clip(max=V - 1)
    return X, z

X_train, z_train = sample_sequences(20000, A, rng)
X_test, z_test = sample_sequences(2000, A, rng)

fig, axes = plt.subplots(1, K, figsize=(12, 2.8))
for k in range(K):
    axes[k].imshow(A[k], cmap='Blues', vmin=0, vmax=1); axes[k].set_title(f'concept {k}: A[{k}]', fontsize=9)
    axes[k].set_xlabel('next token'); axes[k].set_ylabel('current token'); axes[k].grid(False)
plt.tight_layout(); plt.show()
for k in range(K):
    print(f'concept {k}:', ' '.join(map(str, X_test[z_test == k][0][:40])))

---
## Part 1 — The latent-variable view (30 min)

For one sequence $\mathbf{x}=(x_1,\dots,x_T)$ the model is
$$p(\mathbf{x}, z) = p(z)\,p(x_1)\prod_{s=2}^{T}A_z[x_{s-1},x_s], \qquad p(\mathbf{x}) = \sum_{z}p(\mathbf{x},z).$$

**1.1 Posterior over the concept.** After seeing a prefix $x_{1:t}$, Bayes' rule gives
$$p(z\mid x_{1:t}) \;\propto\; p(z)\prod_{s=2}^{t}A_z[x_{s-1},x_s].$$
This is exactly the **responsibility** of a mixture model (the E-step of EM), computed for every prefix length. Implement
`concept_log_posterior` in **log-space** (products of 60 probabilities underflow quickly — compare Problem 1). Use a
running sum over $t$; no loops over sequences.

**1.2 The Bayes-optimal next-token predictor.** Averaging over the unknown concept,
$$p(x_{t+1}=v\mid x_{1:t}) = \sum_{z}p(z\mid x_{1:t})\,A_z[x_t, v].$$
No predictor that sees only $x_{1:t}$ can have a lower expected cross-entropy than this one. Implement `bayes_predictive`.

In [ ]:
def concept_log_posterior(X, A, log_prior):
    """X: (n, T) tokens. Returns L of shape (n, T, K): L[i, t] = log p(z | x_1..x_t) for sequence i
    (t is 0-based: L[i, 0] uses only the first token, which carries no information about z)."""
    # YOUR CODE HERE
    raise NotImplementedError


def bayes_predictive(X, A, log_post):
    """Returns P of shape (n, T-1, V): P[i, t] = p(x_{t+2} = v | x_1..x_{t+1}) (prediction for the next token
    after seeing the first t+1 tokens), averaging the concepts' transition rows with the posterior weights."""
    # YOUR CODE HERE
    raise NotImplementedError


def per_position_ce(P, X):
    """Average cross-entropy (nats) of predicting x_{t+1} at each position t, given predictive distributions P (n, T-1, V)."""
    p_true = np.take_along_axis(P, X[:, 1:, None], axis=2)[..., 0]
    return -np.log(np.maximum(p_true, 1e-300)).mean(0)

In [ ]:
# Tests
L = concept_log_posterior(X_test[:5], A, LOG_PRIOR)
assert L.shape == (5, T, K) and np.allclose(np.exp(L).sum(-1), 1)
assert np.allclose(L[:, 0], LOG_PRIOR)
x = X_test[0, :6]                                       # brute force on one short prefix
brute = np.log(0.25) + sum(np.log(A[:, x[s - 1], x[s]]) for s in range(1, 6))
brute -= np.log(np.exp(brute).sum())
assert np.allclose(L[0, 5], brute)
P = bayes_predictive(X_test[:5], A, L)
assert P.shape == (5, T - 1, V) and np.allclose(P.sum(-1), 1)
print('Part 1 functions: OK')

**1.3 How fast is the concept identified?**

> **Predict first.** After how many tokens do you expect the posterior to put more than 95% of its mass on the true
> concept, for a typical sequence?

Plot, against $t$: (a) the average posterior probability of the true concept, (b) the average entropy
$H = -\sum_z p(z\mid x_{1:t})\log p(z\mid x_{1:t})$ of the posterior (in nats), over the test set.

In [ ]:
# YOUR CODE HERE

---
## Part 2 — Learn the mixture from unlabelled text with EM (45 min)

So far we *knew* the transition matrices. In practice we only have text. The concept of each training sequence is a
latent variable, so the parameters $\theta=\{\pi_k, \mathbf{A}_k\}$ can be estimated by **EM** (Lecture 18), exactly as for a GMM.
Treat the first token as uniform and fixed; it carries no information.

**2.1 Sufficient statistics.** A sequence enters the likelihood only through its **transition counts**
$c_n[a,b]$ = number of times token $a$ is followed by token $b$ in sequence $n$:
$\log p(\mathbf{x}_n\mid z=k) = \text{const} + \sum_{a,b}c_n[a,b]\log A_k[a,b]$. Implement `transition_counts`.

**2.2 EM.** Derive (in a markdown cell, briefly) and implement:
* **E-step:** responsibilities $r_{nk} = p(z_n=k\mid\mathbf{x}_n,\theta) \propto \pi_k\prod_{a,b}A_k[a,b]^{c_n[a,b]}$ — the same computation as
  your `concept_log_posterior` at the last position, now with the current estimates. Work in log-space.
* **M-step:** maximise the expected complete-data log-likelihood. The result is responsibility-weighted counting:
  $\pi_k = \frac1N\sum_n r_{nk}$ and $A_k[a,b] \propto \beta + \sum_n r_{nk}\,c_n[a,b]$ (normalise each row). The small $\beta=0.01$ is a
  MAP-style pseudo-count that keeps every probability positive (Lecture 12).
* Record the **mean log-likelihood per sequence** $\frac1N\sum_n\log p(\mathbf{x}_n\mid\theta)$ at every E-step.

Initialise each $\mathbf{A}_k$ with random rows drawn from `rng.dirichlet(np.ones(V))` and $\pi_k = 1/K$.

In [ ]:
def transition_counts(X):
    """Returns C of shape (n, V, V) with C[i, a, b] = number of transitions a -> b in sequence i."""
    # YOUR CODE HERE
    raise NotImplementedError


def em_mixture_of_chains(C, K, n_iter=50, beta=0.01, seed=0):
    """EM for a mixture of K Markov chains from transition counts C (n, V, V).
    Returns (pi (K,), A (K, V, V), history) with history[i] = mean log-likelihood per sequence at the i-th E-step
    (up to the constant from the first token)."""
    rng_em = np.random.default_rng(seed)
    pi_k = np.full(K, 1.0 / K)
    A_k = rng_em.dirichlet(np.ones(V), size=(K, V))
    history = []
    # YOUR CODE HERE
    raise NotImplementedError
    return pi_k, A_k, history

In [ ]:
# Tests
C_small = transition_counts(X_test[:3])
assert C_small.shape == (3, V, V) and np.all(C_small.sum((1, 2)) == T - 1)
assert C_small[0, X_test[0, 0], X_test[0, 1]] >= 1
pi_t, A_t, h_t = em_mixture_of_chains(transition_counts(X_train[:2000]), K, n_iter=20)
assert np.allclose(A_t.sum(-1), 1) and np.isclose(pi_t.sum(), 1)
assert np.all(np.diff(h_t) > -1e-6), 'EM decreased the log-likelihood'
print('Part 2 functions: OK')

**2.3 Run EM and check what it learned.** Run EM on all of `X_train` from 5 random initialisations (`seed=0..4`), 50
iterations each, and keep the run with the highest final log-likelihood. Then:
1. Plot the log-likelihood per iteration for all 5 runs (one figure).
2. **Match components to concepts.** EM's component 0 need not be concept 0. For each learned component find the true
   concept whose transition matrix is closest (average KL over rows), and print the matching and the distances.
3. Plot the learned (matched) transition matrices next to the true ones.

> **Predict first.** Will all 5 runs reach the same final log-likelihood? Will EM number its components in the same
> order as the true concepts?

In [ ]:
# YOUR CODE HERE

**Question 2.** (a) Write the expected complete-data log-likelihood and show that maximising it gives the M-step above
(a Lagrange multiplier for each row's sum-to-one constraint is enough). (b) Why does EM's numbering of the components
differ from the true one, and why does that not matter for prediction? (c) Did all initialisations agree? What does this
say about EM, and how did you deal with it?

*Write your answer here.*

---
## Part 3 — A tiny language model (25 min)

The cell below defines a 2-layer causal transformer (Lecture 22) with about 100k parameters and trains it by
next-token prediction on `X_train` for 2000 steps. It **never sees the concept labels**. Just run it.

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, d=64, layers=2, heads=4):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(T, d)
        block = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(block, layers, enable_nested_tensor=False)
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, V)
        self.register_buffer('mask', torch.triu(torch.full((T, T), float('-inf')), 1))

    def hidden(self, x):                                  # (B, t) tokens -> (B, t, d) final hidden states
        t = x.shape[1]
        h = self.tok(x) + self.pos(torch.arange(t, device=x.device))
        return self.ln(self.blocks(h, mask=self.mask[:t, :t], is_causal=True))

    def forward(self, x):                                 # next-token logits
        return self.head(self.hidden(x))

def train_lm(X, steps=2000, batch=128, lr=3e-3, seed=0):
    torch.manual_seed(seed)
    model = TinyGPT().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    Xt = torch.from_numpy(X).to(device)
    gen = torch.Generator(device='cpu').manual_seed(seed)
    t0 = time.time()
    for step in range(steps + 1):
        xb = Xt[torch.randint(0, len(Xt), (batch,), generator=gen).to(device)]
        loss = F.cross_entropy(model(xb[:, :-1]).reshape(-1, V), xb[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 500 == 0:
            print(f'step {step:5d}  train cross-entropy {loss.item():.3f}  ({time.time() - t0:.0f}s)')
    return model.eval()

@torch.no_grad()
def lm_predictive(model, X):
    """Next-token distributions of the model, shape (n, T-1, V), as float64 numpy."""
    out = [torch.softmax(model(torch.from_numpy(X[i:i + 500, :-1]).to(device)), -1).double().cpu().numpy()
           for i in range(0, len(X), 500)]
    return np.concatenate(out)

print('parameters:', sum(p.numel() for p in TinyGPT().parameters()))
model = train_lm(X_train)

**3.1 Five predictors on the same footing.** Compute the per-position test cross-entropy of:

1. **Oracle** — *knows* the concept: predicts with $A_{z}[x_t,\cdot]$. Not achievable from the text alone; a lower reference.
2. **Bayes** — your `bayes_predictive` with the *true* $\mathbf{A}$: the best possible predictor that sees only the prefix.
3. **EM mixture** — `bayes_predictive` with *your EM estimates* $(\pi_{\text{EM}}, \mathbf{A}_{\text{EM}})$ from Part 2 (use `np.log(pi_em)`
   as the log-prior).
4. **Transformer** — `lm_predictive(model, X_test)`.
5. **Bigram, no latent variable** — a single transition matrix estimated from all training text (with add-one counts).

Plot the five curves against position, and print the average cross-entropy and the **perplexity**
$\exp(\text{cross-entropy})$ of each. Also print the number of parameters of the EM mixture and of the transformer.

> **Predict first.** Rank the five average cross-entropies. Will the transformer beat the Bayes predictor anywhere? Will
> it beat your EM mixture?

In [ ]:
# YOUR CODE HERE

**Question 3.** (a) Show that for any predictor $q$, its expected cross-entropy at position $t$ equals the Bayes
predictor's cross-entropy plus $\mathbb{E}\big[\mathrm{KL}\big(p_{\text{Bayes}}(\cdot\mid x_{1:t})\,\|\,q(\cdot\mid x_{1:t})\big)\big]$. *Hint:* given the prefix, the next token is
distributed as $p_{\text{Bayes}}$, so the expected loss is $\sum_v p_{\text{Bayes}}(v)\,(-\log q(v))$; add and subtract $\sum_v p_{\text{Bayes}}(v)\log p_{\text{Bayes}}(v)$.
Why can no model trained on this text beat the Bayes curve (up to sampling noise)? (b) Why is the oracle *lower* than
Bayes early on but equal later? (c) What does the gap between the bigram curve and the others tell you about the value
of a latent variable? (d) Compare the EM mixture and the transformer: which is closer to Bayes, and with how many
parameters? Why is this comparison "unfair" in the EM model's favour?

*Write your answer here.*

---
## Part 4 — In-context learning is posterior inference (20 min)

The transformer's cross-entropy *falls* along the sequence — it gets better at predicting this particular sequence as
it reads it, with fixed weights. That is in-context learning. If the transformer is doing what the Bayes predictor does,
its predictive distributions should match $p_{\text{Bayes}}$ **position by position**, not just on average.

Compute $\mathrm{KL}\big(p_{\text{Bayes}}\,\|\,p_{\text{transformer}}\big)$ at each position (average over test sequences) and plot it next to the
posterior entropy from Part 1.3. Also compute the KL of the bigram model, for scale.

In [ ]:
def kl_per_position(P, Q):
    return (P * (np.log(np.maximum(P, 1e-300)) - np.log(np.maximum(Q, 1e-300)))).sum(-1).mean(0)

# YOUR CODE HERE

**Question 4.** (a) Is the transformer close to the Bayes predictor at every position? Where is the gap largest, and
why might that be (compare with the posterior entropy)? (b) In two or three sentences, explain in-context learning in
this world in the language of latent variable models. What plays the role of "the prompt", and what is being
"learned" in context?

*Write your answer here.*

---
## Part 5 — Look inside: is the posterior in the hidden state? (20 min)

If the transformer computes $p(z\mid x_{1:t})$, the posterior should be **readable** from its hidden state at position $t$.
A **linear probe** is just a linear regression trained to predict a quantity of interest from a network's hidden vectors.
Fit one with ridge regression (`sklearn.linear_model.Ridge(alpha=1.0)`, Lecture 7) from the final hidden state
(`model.hidden`, shape $(n, T, 64)$) to the 4-dimensional posterior vector, using hidden states of 1500 test sequences for
fitting and the remaining 500 for evaluation. Report $R^2$ (see the background box) on the held-out sequences for:

1. the trained transformer;
2. the same architecture with **random weights** (`torch.manual_seed(1); TinyGPT()`), as a control;
3. the trained transformer's **input embeddings only** (`model.tok(x) + model.pos(...)`), as a second control.

> **Predict first.** Which of the three will give a high $R^2$?

In [ ]:
@torch.no_grad()
def hidden_states(model, X, embeddings_only=False):
    x = torch.from_numpy(X).to(device)
    if embeddings_only:
        h = model.tok(x) + model.pos(torch.arange(X.shape[1], device=device))
    else:
        h = model.hidden(x)
    return h.cpu().numpy()

# YOUR CODE HERE

**Question 5.** What do the three numbers show? Why is the "input embeddings only" control important? A high probe
$R^2$ shows that the information is *linearly present*; does it prove the network *uses* it to predict? What experiment
would test that?

*Write your answer here.*

---
## Part 6 — Limits, and what changes with more diverse pretraining (25 min)

**6.1 A concept the model has never seen.** Draw a new transition matrix `A_new` the same way, generate 1000 sequences
from it, and compare the per-position cross-entropy of: the oracle (knows `A_new`), the Bayes predictor **over the 4
training concepts**, the transformer, and a simple **in-context counting** predictor that estimates the transition matrix
from the prefix itself.

The counting predictor is itself Bayesian — it is the Beta–Bernoulli posterior predictive of Lecture 12, extended to
$V$ outcomes (see the background box). If each row of the transition matrix has a Dirichlet$(\alpha,\dots,\alpha)$ prior, the
posterior predictive after seeing counts $c_{ab}$ in the prefix is
$$p(x_{t+1}=b\mid x_t=a, x_{1:t}) = \frac{c_{ab}+\alpha}{\sum_{b'}c_{ab'}+V\alpha}.$$
Implement it with $\alpha=0.3$ (the value used to generate the chains).

> **Predict first.** On the new concept, will the transformer behave like the Bayes predictor over the 4 training
> concepts, or like the in-context counting predictor?

In [ ]:
def dirichlet_count_predictive(X, alpha=0.3):
    """Returns P (n, T-1, V): P[i, t] = posterior predictive for the token after x_1..x_{t+1}, from transition counts
    within sequence i only."""
    # YOUR CODE HERE
    raise NotImplementedError

rng_new = np.random.default_rng(123)
A_new = rng_new.dirichlet(0.3 * np.ones(V), size=(1, V))
X_new, _ = sample_sequences(1000, A_new, rng_new)
# YOUR CODE HERE

**6.2 More diverse pretraining.** Now train the same transformer on text in which **every sequence has its own freshly
drawn chain** (unlimited concepts). Run the cell (another 2–4 minutes on CPU), then evaluate the new model on (i) the
new concept `X_new` and (ii) the original test set `X_test`, against the in-context counting predictor and the 4-concept
Bayes predictor.

In [ ]:
def sample_fresh_chains(n, rng):
    As = rng.dirichlet(0.3 * np.ones(V), size=(n, V))
    X = np.zeros((n, T), dtype=np.int64); X[:, 0] = rng.integers(0, V, n)
    for t in range(1, T):
        P = As[np.arange(n), X[:, t - 1]]
        X[:, t] = (P.cumsum(1) < rng.random((n, 1))).sum(1).clip(max=V - 1)
    return X

X_diverse = sample_fresh_chains(20000, np.random.default_rng(7))
model_diverse = train_lm(X_diverse)

In [ ]:
# YOUR CODE HERE

**Question 6.** (a) On the new concept, which predictor does the first transformer resemble, and why can it not adapt
even after 60 tokens? (b) Which predictor does the diversely trained transformer resemble, and what does it give up on
the original 4 concepts? (c) Both transformers are (approximately) Bayes-optimal — under different priors. Explain.
(d) What does this suggest about the role of the *diversity* of an LLM's pretraining data?

*Write your answer here.*

---
## Reflection (answer briefly, 150–250 words in total)
1. Where in this lab did you use the E-step of a latent variable model, and what played the role of the "data point"?
2. A friend says: "In-context learning shows that LLMs learn during inference." Using your results, what would you say?
3. Name one way in which this toy world is too simple to tell us how real LLMs behave.

**Submit** this notebook, executed top to bottom, with your predictions, code, plots and answers.

*References:* S. M. Xie, A. Raghunathan, P. Liang, T. Ma, *An Explanation of In-context Learning as Implicit Bayesian
Inference*, ICLR 2022. A. Raventós, M. Paul, F. Chen, S. Ganguli, *Pretraining Task Diversity and the Emergence of
Non-Bayesian In-context Learning for Regression*, NeurIPS 2023.